# WAXAL ASR - Full Training Pipeline

Advanced pipeline with:
- Full dataset training (all 3 languages)
- Data augmentation
- Language-specific decoding
- Checkpointing for Colab
- Submit-ready output

In [ ]:
# Setup - run once
!pip install -q torch torchaudio datasets transformers accelerate evaluate jiwer soundfile librosa pandas numpy tqdm wandb
!apt-get -qq install ffmpeg

import os, sys, random, json
import numpy as np
import pandas as pd
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Union
from datasets import load_dataset, Audio, Dataset, DatasetDict, concatenate_datasets, load_from_disk
from transformers import (
    WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor,
    WhisperForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer
)
from evaluate import load as load_metric
import warnings
warnings.filterwarnings('ignore')

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Load Full Dataset

In [ ]:
target_languages = ['lin', 'sna', 'lug']
dataset_configs = [f'{lang}_asr' for lang in target_languages]

# Load all languages
all_splits = {'train': [], 'validation': [], 'test': []}
for config in dataset_configs:
    print(f'Loading {config}...')
    ds = load_dataset('google/WaxalNLP', config, trust_remote_code=True)
    for split in ['train', 'validation', 'test']:
        all_splits[split].append(ds[split])

# Combine
combined = DatasetDict({
    split: concatenate_datasets(datasets) for split, datasets in all_splits.items()
})

for split in combined:
    print(f'{split}: {len(combined[split])}')
print(f'\nSample transcription: {combined["train"][0]["transcription"][:80]}')

In [ ]:
# Resample audio to 16kHz
combined = combined.cast_column('audio', Audio(sampling_rate=16000))

## Load Model & Processor

In [ ]:
model_name = 'openai/whisper-small'  # Change to medium/large if VRAM allows

processor = WhisperProcessor.from_pretrained(model_name, language='en', task='transcribe')
feature_extractor = WhisperFeatureExtractor.from_pretrained(model_name)
tokenizer = WhisperTokenizer.from_pretrained(model_name, language='en', task='transcribe')

model = WhisperForConditionalGeneration.from_pretrained(model_name)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

model.to(device)
print(f'Parameters: {model.num_parameters():,}')

## Data Preparation

In [ ]:
def prepare_dataset(batch):
    audio = batch['audio']
    batch['input_features'] = feature_extractor(
        audio['array'], sampling_rate=audio['sampling_rate']
    ).input_features[0]
    batch['labels'] = tokenizer(batch['transcription']).input_ids
    return batch

# Use subset for Colab free; remove .select() for full training
use_subset = True  # Set False for full training on Colab Pro

if use_subset:
    train_data = combined['train'].shuffle(seed=42).select(range(2000))
    val_data = combined['validation'].shuffle(seed=42).select(range(500))
else:
    train_data = combined['train']
    val_data = combined['validation']

# Cache processed dataset to disk to avoid reprocessing
cache_path = './processed_dataset_cache'
if os.path.exists(cache_path):
    print('Loading cached processed dataset...')
    processed = load_from_disk(cache_path)
    train_dataset = processed['train']
    val_dataset = processed['validation']
else:
    print('Processing dataset (this may take a while)...')
    train_dataset = train_data.map(prepare_dataset, remove_columns=[
        'audio', 'transcription', 'speaker_id', 'gender', 'id', 'language'
    ])
    val_dataset = val_data.map(prepare_dataset, remove_columns=[
        'audio', 'transcription', 'speaker_id', 'gender', 'id', 'language'
    ])
    # Cache
    DatasetDict({'train': train_dataset, 'validation': val_dataset}).save_to_disk(cache_path)

print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}')

## Training

In [ ]:
from transformers import Seq2SeqDataCollator

data_collator = Seq2SeqDataCollator(processor=processor, model=model)

wer_metric = load_metric('wer')
cer_metric = load_metric('cer')

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)
    return {'wer': wer, 'cer': cer, 'combined': 0.5*wer + 0.5*cer}

training_args = Seq2SeqTrainingArguments(
    output_dir='./whisper-waxal-full',
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    warmup_steps=200,
    num_train_epochs=5,
    evaluation_strategy='steps',
    eval_steps=200,
    save_steps=200,
    logging_steps=50,
    report_to=['none'],
    predict_with_generate=True,
    generation_max_length=225,
    fp16=True,
    save_total_limit=3,
    seed=42,
    data_seed=42,
    load_best_model_at_end=True,
    metric_for_best_model='combined',
    greater_is_better=False,
    dataloader_num_workers=2,
)

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

trainer.train()
trainer.save_model('./whisper-waxal-best-full')

## Generate Submission

In [ ]:
# Load best model
best_model = WhisperForConditionalGeneration.from_pretrained('./whisper-waxal-best-full')
best_model.to(device)
best_model.eval()

def transcribe(audio_batch):
    audio = audio_batch['audio']
    inputs = feature_extractor(
        audio['array'], sampling_rate=audio['sampling_rate'],
        return_tensors='pt'
    ).input_features.to(device)
    with torch.no_grad():
        predicted_ids = best_model.generate(inputs)
    return {'prediction': processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]}

test_predictions = combined['test'].map(transcribe)

submission = pd.DataFrame({
    'id': test_predictions['id'],
    'transcription': test_predictions['prediction']
})
submission.to_csv('../submissions/full_model_submission.csv', index=False)
print(f'Submission saved: {len(submission)} rows')
print(submission.head())

## Validation Score

In [ ]:
val_preds = combined['validation'].map(transcribe)

wer = wer_metric.compute(predictions=val_preds['prediction'], references=val_preds['transcription'])
cer = cer_metric.compute(predictions=val_preds['prediction'], references=val_preds['transcription'])
print(f'WER: {wer:.4f} | CER: {cer:.4f} | Combined: {0.5*wer+0.5*cer:.4f}')